[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/langchain-ai/langchain-academy/blob/main/module-0/basics.ipynb) [![Open in LangChain Academy](https://cdn.prod.website-files.com/65b8cd72835ceeacd4449a53/66e9eba12c7b7688aa3dbb5e_LCA-badge-green.svg)](https://academy.langchain.com/courses/take/intro-to-langgraph/lessons/56295530-getting-set-up-video-guide)

# LangChain Academy

歡迎來到 LangChain Academy！

## 背景脈絡

在 LangChain，我們的目標是讓打造 LLM 應用變得簡單。你可以打造的其中一種 LLM 應用就是 agent。大家對打造 agent 有非常高的熱情，因為它們能自動化一大堆過去根本做不到的任務。

不過在實務上，要打造一套能可靠執行這些任務的系統，難度高得驚人。在我們陪著使用者把 agent 推上正式環境的過程中，我們學到一件事：很多時候你需要更多的控制權。你可能會希望某個 agent 永遠先呼叫某個特定的 tool，或是根據它目前的 state 套用不同的 prompt。

為了解決這個問題，我們打造了 [LangGraph](https://docs.langchain.com/oss/python/langgraph/overview) —— 一個用來打造 agent 與多代理應用的框架。LangGraph 獨立於 LangChain 套件之外，它的核心設計哲學是幫助開發者在 agent 工作流程中加入更好的精準度與控制力，足以應付真實世界系統的複雜程度。

## 課程架構

本課程由一系列模組（module）構成，每個模組聚焦在一個與 LangGraph 相關的主題。你會看到每個模組各自有一個資料夾，裡面放著一連串的 notebook。每個 notebook 都會搭配一支影片帶你走過這些概念，但這些 notebook 同時也是可以獨立閱讀的 —— 意思是它們本身就含有完整的說明，就算不看影片也能讀懂。每個模組資料夾裡還有一個 `studio` 資料夾，裡面放著一組可以載入 [LangSmith Studio](https://docs.langchain.com/langsmith/quick-start-studio) 的 graph；LangSmith Studio 就是我們用來打造 LangGraph 應用的 IDE。

## 環境設定

開始之前，請先依照 `README` 裡的指示建立環境並安裝相依套件。

## Chat models

在這門課裡，我們會使用 Chat Models（聊天模型）—— 它接收一連串的 message 作為輸入，再回傳 message 作為輸出。LangChain 透過 [第三方整合](https://docs.langchain.com/oss/python/integrations/chat) 支援許多模型。預設情況下，本課程會使用 [ChatOpenAI](https://docs.langchain.com/oss/python/integrations/chat/openai)，因為它既熱門又有不錯的表現。如同前面提到的，請確認你已經備妥 `OPENAI_API_KEY`。

我們先來檢查你的 `OPENAI_API_KEY` 是否已經設定好；如果還沒，系統會請你輸入。

In [1]:
%%capture --no-stderr
%pip install --quiet -U langchain_openai langchain_core langchain_community langchain-tavily

In [2]:
import os, getpass

def _set_env(var: str):
    if not os.environ.get(var):
        os.environ[var] = getpass.getpass(f"{var}: ")

_set_env("OPENAI_API_KEY")

[這裡](https://docs.langchain.com/oss/python/langchain/models) 有一份很實用的 how-to，涵蓋你能用 chat model 做的各種事，不過下面我們會先示範幾個重點。如果你已經照 README 說的跑過 `pip install -r requirements.txt`，那你就已經裝好 `langchain-openai` 套件了。有了它，我們就能實例化（instantiate）出我們的 `ChatOpenAI` 模型物件。各種模型的價格可以在 [這裡](https://openai.com/api/pricing/) 查看。這些 notebook 預設會使用 `gpt-4o`，因為它在品質、價格與速度之間取得了不錯的平衡；當然你也可以改用價格較低的 `gpt-3.5` 系列，或更新的模型。

我們可以為 chat model 設定 [幾個標準參數](https://docs.langchain.com/oss/python/langchain/models#parameters)，其中最常用的兩個是：

* `model`：模型的名稱
* `temperature`：取樣溫度（sampling temperature）

`Temperature` 控制的是模型輸出的隨機程度或創意程度：溫度低（接近 0）時，輸出會更具決定性、更聚焦，適合需要精準度或事實性回答的任務；溫度高（接近 1）時，則適合創意類的任務，或想產生比較多樣化的回答。

In [3]:
from langchain_openai import ChatOpenAI
gpt4o_chat = ChatOpenAI(model="gpt-4o", temperature=0)
gpt35_chat = ChatOpenAI(model="gpt-3.5-turbo-0125", temperature=0)

LangChain 裡的 chat model 內建了不少 [預設方法](https://reference.langchain.com/python/langchain_core/runnables)。大多數情況下，我們會用到的是：

* [stream](https://docs.langchain.com/oss/python/langchain/models#stream)：以串流（stream）的方式一塊一塊回傳回應內容
* [invoke](https://docs.langchain.com/oss/python/langchain/models#invoke)：針對某個輸入呼叫這條 chain

另外，如同前面提到的，chat model 接收 [messages](https://docs.langchain.com/oss/python/langchain/messages) 作為輸入。每個 message 都有一個 role（描述這句話是誰說的）以及一個 content 屬性。這部分我們之後會講得更多，這裡先示範最基本的用法就好。

In [4]:
from langchain_core.messages import HumanMessage

# 建立一則 message
msg = HumanMessage(content="Hello world", name="Lance")

# message 清單
messages = [msg]

# 用一個 message 清單去 invoke 模型
gpt4o_chat.invoke(messages)

AIMessage(content='Hello! How can I assist you today?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 9, 'prompt_tokens': 11, 'total_tokens': 20, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_eb3c3cb84d', 'id': 'chatcmpl-CSWGCGbWLCLdHUAOh9pkNFWl1MWf4', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--529fb9c7-f813-47a7-9662-dea715d4be28-0', usage_metadata={'input_tokens': 11, 'output_tokens': 9, 'total_tokens': 20, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

我們會拿到一個 `AIMessage` 回應。另外要注意的是，我們其實可以直接用一個字串去 invoke 一個 chat model。當你傳入一個字串作為輸入時，它會被轉換成一個 `HumanMessage`，然後再傳給底層的模型。


In [5]:
gpt4o_chat.invoke("hello world")

AIMessage(content='Hello! How can I assist you today?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 9, 'prompt_tokens': 9, 'total_tokens': 18, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_cbf1785567', 'id': 'chatcmpl-CSWGCXlVYTEWoHAFm3GXIIgUO1gCb', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--43b070a6-7676-4aa1-984c-c0cd43d45c1e-0', usage_metadata={'input_tokens': 9, 'output_tokens': 9, 'total_tokens': 18, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [6]:
gpt35_chat.invoke("hello world")

AIMessage(content='Hello! How can I assist you today?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 9, 'prompt_tokens': 9, 'total_tokens': 18, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-CSWGDY5KePqihRcWDX3IGEZfSoyld', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--f7963c03-f8b2-4eba-bc7f-4d58520fa661-0', usage_metadata={'input_tokens': 9, 'output_tokens': 9, 'total_tokens': 18, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

這個介面在所有 chat model 之間都是一致的，而且模型通常只在每個 notebook 一開始時初始化一次。

所以，如果你對某個其他供應商有強烈偏好，你可以輕鬆地在不同模型之間切換，而完全不需要更動下游的程式碼。


## Search Tools

你在 README 裡也會看到 [Tavily](https://tavily.com/)，它是一個專為 LLM 與 RAG 最佳化的搜尋引擎，主打高效、快速且穩定持久的搜尋結果。如同前面提到的，註冊很簡單，而且它提供了相當大方的免費額度。有些課程內容（在 Module 4）會預設使用 Tavily，當然，如果你想自己改寫程式碼，也可以換用其他的搜尋工具。

In [7]:
_set_env("TAVILY_API_KEY")

In [11]:
from langchain_tavily import TavilySearch  # 在 1.0 版更新

tavily_search = TavilySearch(max_results=3)

data = tavily_search.invoke({"query": "What is LangGraph?"})
search_docs = data.get("results", data)

In [12]:
search_docs

[{'url': 'https://www.datacamp.com/tutorial/langgraph-tutorial',
  'title': 'LangGraph Tutorial: What Is LangGraph and How to Use It?',
  'content': 'LangGraph is a library within the LangChain ecosystem that provides a framework for defining, coordinating, and executing multiple LLM agents (or chains) in a structured and efficient manner. By managing the flow of data and the sequence of operations, LangGraph allows developers to focus on the high-level logic of their applications rather than the intricacies of agent coordination. Whether you need a chatbot that can handle various types of user requests or a multi-agent system that performs complex tasks, LangGraph provides the tools to build exactly what you need. LangGraph significantly simplifies the development of complex LLM applications by providing a structured framework for managing state and coordinating agent interactions.',
  'score': 0.9581988,
  'raw_content': None},
 {'url': 'https://www.geeksforgeeks.org/machine-learning